In [1]:
import sys
sys.path.append('..')

In [5]:
import os
import pickle
from collections import OrderedDict

import numpy as np
import pandas as pd
import scipy as sp

# import tensorflow as tf
# from tensorflow.keras import Model, Input, losses, layers, optimizers

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import MinMaxScaler

from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm
import seaborn as sns

import itertools

from src.datasets import wilt
from src.evaluation import data_benchmark
# from src.models import vae_keras
# from src.datasets import adults#, dataset_utils

# Load the original data

In [7]:
x_num_train = np.load('../data/wilt/X_num_train.npy', allow_pickle=True)
y_train = np.load('../data/wilt/y_train.npy', allow_pickle=True).reshape(-1,1)
data_train = np.concatenate([x_num_train, y_train], axis=1)

x_num_test = np.load('../data/wilt/X_num_test.npy', allow_pickle=True)
y_test = np.load('../data/wilt/y_test.npy', allow_pickle=True).reshape(-1,1)
data_test = np.concatenate([x_num_test, y_test], axis=1)

x_num_val = np.load('../data/wilt/X_num_val.npy', allow_pickle=True)
y_val = np.load('../data/wilt/y_val.npy', allow_pickle=True).reshape(-1,1)
data_val = np.concatenate([x_num_val, y_val], axis=1)

data_orig = np.concatenate((data_train, data_test, data_val), axis=0)
data_orig = pd.DataFrame(data_orig)
print(data_orig)

cols_to_normalize = data_orig.columns[:5]  # columns 0, 1, 2, 3, 4
scaler = MinMaxScaler()
data_orig[cols_to_normalize] = scaler.fit_transform(data_orig[cols_to_normalize])

print(data_orig)


               0           1           2           3          4    5
0     108.134454  199.333333  107.066667  253.000000  20.509077  0.0
1     124.366957  202.898148   92.175926  459.935185  25.997314  0.0
2     107.978947  217.958333   93.583333  677.500000  25.557635  0.0
3     127.745434  209.181818   96.381818  570.200000  41.612236  0.0
4     126.588894  362.300000  273.571429  610.907143  37.734299  0.0
...          ...         ...         ...         ...        ...  ...
4834  126.977735  258.442983  167.368421  231.127193  19.945492  0.0
4835  115.145221  191.911765   79.117647  301.176471  19.904378  0.0
4836  134.836760  238.029703  128.970297  464.841584  33.133196  0.0
4837  118.919028  200.910256   86.269231  565.384615  17.675782  0.0
4838  113.647321  222.428571  149.857143  530.714286  34.420939  1.0

[4839 rows x 6 columns]
             0         1         2         3         4    5
0     0.589992  0.047423  0.036585  0.110204  0.131041  0.0
1     0.678558  0.049482  0

# Load the synthetic data (vanilla aka ddpm_cb_best) and run the evaluation on the vanilla synthetic data

In [16]:
x_num_train = np.load('../exp/wilt/ddpm_cb_best/X_num_train.npy', allow_pickle=True)
y_trian = np.load('../exp/wilt/ddpm_cb_best/y_train.npy', allow_pickle=True).reshape(-1,1)

data_synth = np.concatenate([x_num_train, y_trian], axis=1)
data_synth = pd.DataFrame(data_synth)

print(data_synth)

cols_to_normalize = data_synth.columns[:5]  # columns 0, 1, 2, 3, 4
scaler = MinMaxScaler()
data_synth[cols_to_normalize] = scaler.fit_transform(data_synth[cols_to_normalize])

print(data_synth)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth, [1,1,1,1])

               0           1           2           3          4    5
0     113.400658  219.265809   99.492330  540.157308  11.919928  0.0
1     118.852243  200.674723   86.893198  461.974412  34.524832  0.0
2     136.773325  211.951730   91.461666  457.971144  10.723522  0.0
3     131.511699  352.173927  247.454545  515.007036  34.285630  0.0
4     110.028419  215.152981   97.213642  600.984100  20.202493  0.0
...          ...         ...         ...         ...        ...  ...
3091  133.604966  271.095345  160.783518  561.366898  16.798442  0.0
3092  109.958113  217.056123   96.433270  537.104801  14.449361  0.0
3093  127.665353  199.702163   87.713191  476.666518  26.094449  0.0
3094  113.957436  229.330529  140.509267  527.330277  19.107713  1.0
3095  134.258249  193.810200   87.536079  405.440835  21.243281  0.0

[3096 rows x 6 columns]
             0         1         2         3         4    5
0     0.618725  0.035548  0.022566  0.300270  0.100288  0.0
1     0.648469  0.024545  0

TypeError: evaluate_synthetic_data() missing 2 required positional arguments: 'cat_cols' and 'metrics'

In [17]:
## testing some shit
cont_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
cat_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship',
            'race', 'sex', 'country', 'income']

x_cat = np.load('../exp/adult/ddpm_cb_dp/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/ddpm_cb_dp/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/ddpm_cb_dp/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)


x_train = np.concatenate([x_cat, y], axis=1)
print(x_train, "NEWLINE \n")
data_synth = np.concatenate([x_train, x_num], axis=1)
# print(data_synth, "NEWLINE \n")
data_synth = pd.DataFrame(data_synth, columns=cat_cols+cont_cols)
# # print(data_synth, "NEWLINE \n")

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth['income'] = data_synth['income'].map(map_dict)
print(data_synth, "NEWLINE \n")

# apply same preprocessing as is applied to original raw adult set
data_synth, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = wilt.load_data(data_synth)
print(data_synth, "NEWLINE \n")


print(data_synth["age"].unique().tolist(), "NEWLINE \n")


# align columns with original dataset
# data_orig, data_synth = data_orig.align(data_synth, join="left", axis=1)

# # evaluate
# data_benchmark.evaluate_synthetic_data(data_orig, data_synth, cont_cols, cat_cols, [1,1,1,1])

[['Private' 'HS-grad' 'Never-married' ... 'Male' 'United-States' 0]
 ['Private' 'HS-grad' 'Married-civ-spouse' ... 'Male' 'United-States' 0]
 ['Private' 'HS-grad' 'Married-civ-spouse' ... 'Male' 'United-States' 1]
 ...
 ['Private' 'HS-grad' 'Never-married' ... 'Male' 'United-States' 0]
 ['Private' 'HS-grad' 'Married-civ-spouse' ... 'Male' 'United-States' 1]
 ['Private' 'HS-grad' 'Married-civ-spouse' ... 'Male' 'United-States' 0]] NEWLINE 

       workclass  education      marital-status       occupation  \
0        Private    HS-grad       Never-married     Craft-repair   
1        Private    HS-grad  Married-civ-spouse     Craft-repair   
2        Private    HS-grad  Married-civ-spouse  Exec-managerial   
3        Private    HS-grad  Married-civ-spouse   Prof-specialty   
4        Private  Bachelors       Never-married   Prof-specialty   
...          ...        ...                 ...              ...   
215995   Private    HS-grad       Never-married    Other-service   
215996   Pri

# Load the synthetic dataset (DP) and run the evaluation on the dp synthetic data

In [32]:
x_cat = np.load('../exp/adult/dp/5-e500s0.2/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/dp/5-e500s0.2/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/dp/5-e500s0.2/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)

x_train = np.concatenate([x_cat, y], axis=1)
data_synth_dp = np.concatenate([x_train, x_num], axis=1)
data_synth_dp = pd.DataFrame(data_synth_dp, columns=cat_cols+cont_cols)

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth_dp['income'] = data_synth_dp['income'].map(map_dict)

# apply same preprocessing as is applied to original raw adult set
data_synth_dp, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = adults.load_adults_data(raw_input_data=data_synth_dp)    

# align columns with original dataset
data_orig, data_synth_dp = data_orig.align(data_synth_dp, join="left", axis=1)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth_dp, cont_cols, cat_cols, [1,1,1,1])

FileNotFoundError: [Errno 2] No such file or directory: '../exp/adult/dp/5-e500s0.2/X_cat_train.npy'

In [ ]:
x_cat = np.load('../exp/adult/dp/6-e100s0.34/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/dp/6-e100s0.34/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/dp/6-e100s0.34/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)

x_train = np.concatenate([x_cat, y], axis=1)
data_synth_dp = np.concatenate([x_train, x_num], axis=1)
data_synth_dp = pd.DataFrame(data_synth_dp, columns=cat_cols+cont_cols)

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth_dp['income'] = data_synth_dp['income'].map(map_dict)

# apply same preprocessing as is applied to original raw adult set
data_synth_dp, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = adults.load_adults_data(raw_input_data=data_synth_dp)    

# align columns with original dataset
data_orig, data_synth_dp = data_orig.align(data_synth_dp, join="left", axis=1)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth_dp, cont_cols, cat_cols, [1,1,1,1])

In [19]:
x_cat = np.load('../exp/adult/dp/7-e50s0.42/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/dp/7-e50s0.42/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/dp/7-e50s0.42/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)

x_train = np.concatenate([x_cat, y], axis=1)
data_synth_dp = np.concatenate([x_train, x_num], axis=1)
data_synth_dp = pd.DataFrame(data_synth_dp, columns=cat_cols+cont_cols)

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth_dp['income'] = data_synth_dp['income'].map(map_dict)

# apply same preprocessing as is applied to original raw adult set
data_synth_dp, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = adults.load_adults_data(raw_input_data=data_synth_dp)    

# align columns with original dataset
data_orig, data_synth_dp = data_orig.align(data_synth_dp, join="left", axis=1)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth_dp, cont_cols, cat_cols, [1,1,1,1])

FileNotFoundError: [Errno 2] No such file or directory: '../exp/adult/dp/7-e50s0.42/X_cat_train.npy'

In [20]:
x_cat = np.load('../exp/adult/dp/8-e10s0.66/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/dp/8-e10s0.66/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/dp/8-e10s0.66/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)

x_train = np.concatenate([x_cat, y], axis=1)
data_synth_dp = np.concatenate([x_train, x_num], axis=1)
data_synth_dp = pd.DataFrame(data_synth_dp, columns=cat_cols+cont_cols)

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth_dp['income'] = data_synth_dp['income'].map(map_dict)

# apply same preprocessing as is applied to original raw adult set
data_synth_dp, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = adults.load_adults_data(raw_input_data=data_synth_dp)    

# align columns with original dataset
data_orig, data_synth_dp = data_orig.align(data_synth_dp, join="left", axis=1)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth_dp, cont_cols, cat_cols, [1,1,1,1])

FileNotFoundError: [Errno 2] No such file or directory: '../exp/adult/dp/8-e10s0.66/X_cat_train.npy'

In [21]:
x_cat = np.load('../exp/adult/dp/9-e1s2.58/X_cat_train.npy', allow_pickle=True)
x_num = np.load('../exp/adult/dp/9-e1s2.58/X_num_train.npy', allow_pickle=True)
y = np.load('../exp/adult/dp/9-e1s2.58/y_train.npy', allow_pickle=True)
y = y.reshape(-1,1)

x_train = np.concatenate([x_cat, y], axis=1)
data_synth_dp = np.concatenate([x_train, x_num], axis=1)
data_synth_dp = pd.DataFrame(data_synth_dp, columns=cat_cols+cont_cols)

# preprocess data produced from DM to match the original dataset, for evaluation

# income needs to be transformed to strings to match original raw adult set
map_dict = {1: '>50K', 0: '<=50K'}
data_synth_dp['income'] = data_synth_dp['income'].map(map_dict)

# apply same preprocessing as is applied to original raw adult set
data_synth_dp, enc_dict_synth, dec_dict_synth, Scaler_synth, cont_cols, cat_cols = adults.load_adults_data(raw_input_data=data_synth_dp)    

# align columns with original dataset
data_orig, data_synth_dp = data_orig.align(data_synth_dp, join="left", axis=1)

# evaluate
data_benchmark.evaluate_synthetic_data(data_orig, data_synth_dp, cont_cols, cat_cols, [1,1,1,1])

FileNotFoundError: [Errno 2] No such file or directory: '../exp/adult/dp/9-e1s2.58/X_cat_train.npy'